In [1]:
from config import *
import Minimal_Model_1D as mm1D
import Minimal_Model_2D as mm2D
import Operators2D as O2d

In [9]:
def Correlation_2D(A, ax, ay, i, ip, j, jp, dl=1):

    """
        Returns correlation function between point labelled by i, j on a n X m lattice
        
        Args:
            A (float) : normalizing ocnstant for correlation function
            i, j (int) : index of two poins between which correlation function needs to be calculated
            dl (float) : decay length of correlation function 
        
    """

    #x_dis = ((i%n) - (j%n))*ax   ## x distance between two points on lattice with index i,j labelled in graph way 

    #y_dis = ((i//n) - (j//n))*ay   ## y distance between two points on lattice with index i,j labelled in graph way 

    x_dis = (i -ip)*ax   ## x distance between two points on lattice with index i,j labelled in graph way 

    y_dis = (j -jp)*ay   ## y distance between two points on lattice with index i,j labelled in graph way 

    dis_sq = x_dis**2 + y_dis**2

    lmbda_sq = dl**2*(ax**2+ay**2)

    return A**2 * math.exp(-dis_sq/lmbda_sq)

In [10]:
def QW_wf_2d(n, m, i, ax, ay):

    V = np.empty((m*n))

    for i in range(m):

        for j in range(n):

            #V[i*n+j] = math.sqrt(2/(n-1))*math.sqrt(2/(m-1))*math.sin(j*math.pi/(n-1))*math.sin(i*math.pi/(m-1))

            V[i*n+j] = math.sin(j*math.pi/(n))*math.sin(i*math.pi/(m))*math.sqrt(2/(n)/ax)*math.sqrt(2/(m)/ay)

    return V 

In [11]:
### Now checking for 4MN X 4MN wavevector 
### Now extending the function to integrate over wavefunction with all degrees of freedom i.e spin and particle

def variance_of_equivalent_uncorrelated_disorder(WV, A, n, m, ax, ay, dl=1):

    """
        Returns equivalent <VV> or variance for uncorrelated disorder in 2D given correlated disorder magnitude A   
        
        Args:
            WV (array) : Array of dim (nXm, 1) representing wavefunction 
            A (float) : normalizing ocnstant for correlation function
            dl (float) : decay length of correlation function 
        
    """
 
    WVb = WV.reshape((4,m,n))           ### reshaping one D wavevector for ease of summing with the correlation function which relies on i,j being latice index

    var_corr = np.empty((4*m*n, 4*m*n))                        ### intializing variance of uncorrelated disorder

    for k in range(4):

        for kp in range(4):

            for i in range(n):

                for j in range(m):

                    for ip in range(n):

                        for jp in range(m):

                            ### summing square of given wavefunction with correlation function to find corr_var

                            var_corr[i+j*n+k*m*n, ip+jp*n+kp*m*n] = abs(WVb[k,j,i])**2*abs(WVb[kp,jp,ip])**2*Correlation_2D(A, ax, ay, i, ip, j, jp, dl)
                            

    ### Please note, take the ax**2*ay**2 (area of single element of integral), 
    ### include it either in normalization of wavefunction, or in variance reimann sum calculation. 
    ### Do not include it at both the places.

    var_corr_sum = np.sum(var_corr)*ax**2*ay**2

    var_uncorr = np.empty((m*n, 4, 4))

    for k in range(4):

            for kp in range(4):

                for j in range(m):

                    for i in range(n):

                        ### summing square of given wavefunction with correlation function to find corr_var

                        var_uncorr[i+j*n, k, kp] = abs(WVb[k,j,i])**2*abs(WVb[kp,j,i])**2

    factor = np.sum(var_uncorr)*ax*ay

    eqv_uncorr_sum = var_corr_sum/ factor

    return eqv_uncorr_sum, var_corr_sum, factor

def QW_wf_2d(n, m, i, ax, ay):

    V = np.empty((m*n))

    for i in range(m):

        for j in range(n):

            #V[i*n+j] = math.sqrt(2/(n-1))*math.sqrt(2/(m-1))*math.sin(j*math.pi/(n-1))*math.sin(i*math.pi/(m-1))

            V[i*n+j] = math.sin(j*math.pi/(n))*math.sin(i*math.pi/(m))*math.sqrt(2/(n)/ax)*math.sqrt(2/(m)/ay)

    return V 

In [12]:
N = 50

AX = 0.2

M = 50

AY = 0.2

D = 5

print('LMBDA_SQ = ', (AX**2+AY**2)*D**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)

print(S.shape)

print(np.sum(np.abs(S)**2))

print(np.sum(np.abs(S)**2)*AX*AY)

print(variance_of_equivalent_uncorrelated_disorder(S,1,N,M,AX,AY,D))



LMBDA_SQ =  2.0000000000000004
(10000,)
100.0
4.0
(5.549406136135906, 1.9977862090089256, 0.3599999999999999)


In [14]:
N = 50

AX = 0.2

M = 50

AY = 0.2

D = 5

print('LMBDA_SQ = ', (AX**2+AY**2)*D**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

print(S.shape)

print(np.sum(np.abs(S)**2))

print(np.sum(np.abs(S)**2)*AX*AY)

print(variance_of_equivalent_uncorrelated_disorder(S,1,N,M,AX,AY,D))


LMBDA_SQ =  2.0000000000000004
(10000,)
25.0
1.0
(5.549406136135906, 0.12486163806305785, 0.022499999999999992)


In [15]:
N = 20

AX = 0.5

M = 20

AY = 0.5

D = 2

print('LMBDA_SQ = ', (AX**2+AY**2)*D**2)

S = np.tile(QW_wf_2d(N, M, 1, AX, AY),4)/2

print(S.shape)

print(np.sum(np.abs(S)**2))

print(np.sum(np.abs(S)**2)*AX*AY)

print(variance_of_equivalent_uncorrelated_disorder(S,1,N,M,AX,AY,D))

LMBDA_SQ =  2.0
(1600,)
3.9999999999999996
0.9999999999999999
(5.549449274917806, 0.12486260868565062, 0.0225)
